# Pagination

Use `Page[Model]` and `paginate()` to slice through graph results.

## Model & test data

In [1]:
from rdflib import RDF, XSD, Graph, Literal, Namespace

from rdfantic import GraphModel, paginate, predicate

SCHEMA = Namespace("http://schema.org/")
EX = Namespace("http://example.org/")


class BookView(GraphModel):
    rdf_type = SCHEMA["Book"]
    name: str = predicate(SCHEMA["name"])
    author: str | None = predicate(SCHEMA["author"])


# Build a graph with 15 books
g = Graph()
for i in range(15):
    subj = EX[f"book{i:02d}"]
    g.add((subj, RDF.type, SCHEMA["Book"]))
    g.add((subj, SCHEMA["name"], Literal(f"Book {i}", datatype=XSD.string)))
    g.add((subj, SCHEMA["author"], Literal(f"Author {i % 5}", datatype=XSD.string)))

## Paginate through all books

In [2]:
page_size = 5
offset = 0

while True:
    page = paginate(BookView, g, offset=offset, limit=page_size)
    page_num = offset // page_size + 1
    print(f"Page {page_num} (offset={page.offset}, total={page.total}):")

    for book in page.items:
        print(f"  {book.name} by {book.author}")

    offset += page_size
    if offset >= page.total:
        break
    print()

Page 1 (offset=0, total=15):
  Book 0 by Author 0
  Book 1 by Author 1
  Book 2 by Author 2
  Book 3 by Author 3
  Book 4 by Author 4

Page 2 (offset=5, total=15):
  Book 5 by Author 0
  Book 6 by Author 1
  Book 7 by Author 2
  Book 8 by Author 3
  Book 9 by Author 4

Page 3 (offset=10, total=15):
  Book 10 by Author 0
  Book 11 by Author 1
  Book 12 by Author 2
  Book 13 by Author 3
  Book 14 by Author 4


## Serialization for JSON responses

In [3]:
first_page = paginate(BookView, g, offset=0, limit=3)
data = first_page.model_dump()
print(f"Serialized page keys: {list(data.keys())}")
print(f"Items in first page: {len(data['items'])}")
print(f"Total: {data['total']}")

Serialized page keys: ['items', 'total', 'offset', 'limit']
Items in first page: 3
Total: 15
